<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="05_nemo_agent_toolkit_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a href="03_low_level_mcp_ja.ipynb">3</a>
        <a href="04_langraph_agent_ja.ipynb">4</a>
        <a href="05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a >6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="bonus_challenge/07_bonus_challenge_ja.ipynb">Next Notebook</a></span>
</div>

# チャレンジ概要
---

現代の e-commerce サポートは agentic system にとって有力なユースケースです。顧客は幅広い製品に関する質問をしたり、過去の購入についてのサポートを求めたり、返金や他のアカウントサポートが必要な際は素早く正確なルーティングを期待します。本チャレンジでは、これらのニーズを 1 つの実用的なワークフローにまとめ、現実的なカスタマーサポートの場で講座のコンセプトを応用できるようにしています。  

このコースチャレンジでは、提供されている music store サポートシステムを拡張し、顧客のニーズを理解して各リクエストを適切なワークフローにルーティングするインテリジェントな e-commerce agent を構築します。Agent は一般的な顧客問い合わせに回答し、返金リクエストを処理し、顧客の intent を判別して適切なサポート体験を提供する必要があります。  

本チャレンジを通じて、参加者は以下ができるようになります：
1) low level の Python SDK を使って MCP サーバーを作成する
2) MCP サーバーを任意の MCP クライアントが再利用できるフォーマットでパッケージ化する
3) progressive disclosure に基づいた agent skill を実装する
4) langgraph を通じた human-in-the-loop を含むワークフローを構築する
5) agentic ワークフローの LLM バックボーンとして NVIDIA NIM を活用する

<left>
<p>元のワークフローは次のとおりです <a href="https://docs.smith.langchain.com/evaluation/tutorials/agents">langchain agent evaluation</a>
</p>
<img src="./images/orig_workflow.png" width="500"/>
</left>

このワークフローの主なコンポーネントは 3 つです：
1) intent classifier、
2) 一般的な顧客問い合わせを処理する QnA agent、
3) 返金リクエストを処理する refund agent。

このチャレンジを完了するために、以下を実装します：
1) intent が不明な場合に人間が介入できるようにする
2) refund agent に MCP プロトコルを組み込んで invoice データにアクセスできるようにする
3) qna agent で agent skill を活用し、根拠に基づいた customer support レスポンスを生成する

<left>
<p>最終的なワークフロー</p>
<img src="images/new_workflow.jpeg" width="500"/>
</left>

Refund agent は invoice MCP サーバーに接続して、invoice と返金に関する顧客クエリに回答します。

Invoice MCP サーバーは以下のツールへのアクセスを提供します：
1) invoice 検索
2) invoice 返金

QnA agent は agent skill を搭載しており、Chinook DB に対して実行できる SQL クエリを生成し、アーティスト、アルバム、トラックに関する情報を取得して顧客向けの回答を生成します。

<left>
<p>invoice/qna agent のコンポーネント</p>
<img src="images/final_workflow.jpeg" width="1000"/>
</left>

## はじめに

すべてのチャレンジ素材は `challenge/` 配下にあります。各ステージは前のステージを基盤としているため、課題を順番に進めるのが最もスムーズな取り組み方です。

まずはこれらの編集可能なファイルから始めてください：

1. Invoice MCP サーバー: [mcp-servers/invoice/src/mcp_server_invoice/server_http.py](../../challenge/mcp-servers/invoice/src/mcp_server_invoice/server_http.py)
2. QnA agent の skill: [qna_agent/skills/music-store-assistant/SKILL.md](../../challenge/qna_agent/skills/music-store-assistant/SKILL.md)
3. End-to-end LLM ワークフロー: [llm_workflow/main.py](../../challenge/llm_workflow/main.py) と [llm_workflow/mcp_http_client.py](../../challenge/llm_workflow/mcp_http_client.py)
4. NeMo Agent Toolkit の設定: [nemo_agent_toolkit/workflow.yaml](../../challenge/nemo_agent_toolkit/workflow.yaml)

各チャレンジフォルダーには `README.md` がありセットアップの詳細が記載されています。また、対応する test フォルダーには提出前にローカルでソリューションを検証する方法が示されています。

---

## 課題

1. 各チャレンジフォルダーには、すべての依存関係を記載した `pyproject.toml` ファイルがあります。これらの依存関係のみを使用してください。
2. ファイル/フォルダー名は変更しないでください。
3. すべての Python ファイルにおいて、import 文、環境変数、クラス/関数のヘッダー、モデル ID は変更しないでください。

### 課題 1 - Invoice MCP サーバー

この課題は***5点***満点で採点されます。

invoice MCP サーバーのフォルダー/ファイルが以下のとおり提供されています。

```bash
invoice
├── data
│   └── chinook.db
├── pyproject.toml
├── README.md
├── src
│   └── mcp_server_invoice
│       ├── __init__.py
│       └── server_http.py
└── uv.lock
```

以下の Python ファイルのコードを完成させてください。
- [invoice MCP サーバー](../../challenge/mcp-servers/invoice/src/mcp_server_invoice/server_http.py)

### ヒント: JSON Schema 書き方リファレンス

03 cell [5] で見た `{"type": "integer"}` 以外に、invoice tool では以下も使います:

- **array (= list)**: `{"type": "array", "items": {"type": "integer"}}`
- **nullable (= Optional)**: `{"type": ["string", "null"]}`
- **boolean**: `{"type": "boolean"}`

詳細は [JSON Schema docs](https://json-schema.org/learn/getting-started-step-by-step) 参照。

#### ソリューションのテスト

提出前にソリューションをローカルでテストするために、以下を使用できます。テストケースは最小限のものであり、独自にさらに追加してください。
- [テスト手順](../../challenge/mcp-server-invoice-test/README.md)

---

### 課題 2 - QNA agent

この課題は***2点***満点で採点されます。

QNA agent のフォルダー/ファイルが以下のとおり提供されています。

```bash
qna_agent
├── main.py
├── notobvious.db
├── pyproject.toml
├── README.md
├── skills
│   └── music-store-assistant
│       └── SKILL.md
├── skills_ref
│   ├── errors.py
│   ├── models.py
│   ├── parser.py
│   ├── prompt.py
│   ├── utils.py
│   └── validator.py
└── uv.lock
```

以下の Python ファイルのコードを完成させてください。
- [QNA agent](../../challenge/qna_agent/skills/music-store-assistant/SKILL.md)

### ヒント: データベーススキーマの確認

`notobvious.db` のテーブル構造を SKILL.md に書く際は、notebook 上で以下を実行して schema を確認してください:

```python
!sqlite3 ../../challenge/qna_agent/notobvious.db ".schema"
```

テーブル一覧だけ見るなら `.tables`、特定テーブルだけなら `.schema <table_name>` も使えます。

提出前にソリューションをローカルでテストするために、以下を使用できます。テストケースは最小限のものであり、独自にさらにテストケースを追加してください。
- [テスト手順](../../challenge/qna-agent-test/README.md)

---

### 課題 3 - LLM ワークフロー

*この課題を開始する前に、課題 1 と 2 を完了してください*

この課題は***4点***満点で採点されます。

LLM ワークフローのフォルダー/ファイルが以下のとおり提供されています。

```bash
llm_workflow
├── __init__.py
├── chinook.db
├── main.py
├── mcp_http_client.py
├── pyproject.toml
├── README.md
└── uv.lock
```

以下の Python ファイルのコードを完成させてください。

- [llm_workflow/main.py](../../challenge/llm_workflow/main.py)
- [llm_workflow/mcp_http_client.py](../../challenge/llm_workflow/mcp_http_client.py)

#### ソリューションのテスト

提出前にソリューションをローカルでテストするために、以下を使用できます。テストケースは最小限のものであり、独自にさらにテストケースを追加してください。  
- [テスト手順](../../challenge/llm-workflow-test/README.md)

---

### 課題 4 - NeMo Agent Toolkit

*この課題を開始する前に、課題 1、2、3 を完了してください*

この課題は***5点***満点で採点されます。

この課題では、invoice MCP サーバーに接続して Chinook データベースの invoice に関するクエリに回答する [NeMo Agent Toolkit (NAT)](https://docs.nvidia.com/nemo/agent-toolkit/latest/index.html) 用の `workflow.yaml` 設定を作成します。

ワークフローには以下を定義する必要があります：
1. invoice MCP サーバーに接続する **MCP クライアント** function group
2. NVIDIA NIM endpoint でバックアップされた **LLM**
3. MCP ツールと LLM を使って invoice 関連の質問に回答する **ReAct agent** ワークフロー

評価では、`workflow.yaml` に対して 5 つのテスト問題が実行されます。各問題は 0 または 1 でスコアリングされます。

NeMo Agent Toolkit チャレンジのフォルダー/ファイルが以下のとおり提供されています。

```bash
nemo_agent_toolkit
├── README.md
└── workflow.yaml
```

以下のファイルのワークフロー設定を完成させてください。
- [nemo_agent_toolkit/workflow.yaml](../../challenge/nemo_agent_toolkit/workflow.yaml)

#### ソリューションのテスト

invoice MCP サーバーを起動してから以下を実行することで、ワークフローをローカルでテストできます：

```bash
nat run --config_file workflow.yaml --input "<question>"
```

---

## 提出

1. 以下のスクリプトを実行して [submission.zip](../../challenge/create-zip.sh) を作成してください

```bash
./create-zip.sh
```

In [ ]:
# Notebook から実行する場合 (= terminal 不要)
!cd ../../challenge && bash create-zip.sh && ls -la submission.zip

`submission.zip` には必要な編集可能な Python ファイルのみが含まれます。zip アーカイブのフォルダー構成は以下のとおりです。

```bash
submission
├── llm-workflow
│   ├── main.py
│   └── mcp_http_client.py
├── mcp-servers
│   └── invoice
│       └── server_http.py
├── nemo-agent-toolkit
│   └── workflow.yaml
└── qna_agent
    └── SKILL.md
```

2. zip アーカイブをアップロードしてソリューションを提出してください。

---

## ライセンス

Copyright © 2026 OpenACC-Standard.org. 本資料は OpenACC-Standard.org が NVIDIA Corporation との協力のもと、Creative Commons Attribution 4.0 International (CC BY 4.0) ライセンスのもとで公開しています。本資料には他の団体が開発したハードウェアおよびソフトウェアへの参照が含まれており、該当するすべてのライセンスおよび著作権が適用されます。

<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="05_nemo_agent_toolkit_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a href="03_low_level_mcp_ja.ipynb">3</a>
        <a href="04_langraph_agent_ja.ipynb">4</a>
        <a href="05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a >6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="bonus_challenge/07_bonus_challenge_ja.ipynb">Next Notebook</a></span>
</div>